# 01 - EDA: Bangladesh Electricity Demand 2016-2024

Hourly dataset columns: `Datetime, Temperature(2m), Rhumadity(2m), SPressure(kPa), Population, GDP, Generation(MW), Demand(MW)`.

Key findings (full report generated by `python src/eda.py`, saved to `results/data_quality.txt`):

1. 78,912 rows, complete hourly grid, no gaps, no missing values.
2. **`Generation(MW)` equals `Demand(MW)` exactly in 56,002/78,912 rows (71%)** -> the column is a copy artifact, so no power-balance physics term uses it; modeling targets `Demand(MW)` only.
3. 74 isolated sensor glitches (e.g. a single hour at 143 MW inside a 13.5 GW series); cleaned with median-ratio + jump-reversal detection and interpolation.
4. Demand has a strong thermal driver: 64.6% of hours are above the 25 C comfort threshold, motivating the PINN thermal-sensitivity constraint dDemand/dT >= 0 in the hot regime.
5. Annual peak grows 9.0 GW (2016) -> 17.2 GW (2024), tracking GDP/population static covariates.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from config import Config
import data_pipeline as dp

cfg = Config()
df = dp.load_frame(cfg.csv_path)
df[['Temperature(2m)', 'Rhumadity(2m)', 'SPressure(kPa)', 'demand_clean']].describe()

In [ ]:
# physics-flavoured features used by all models (CDH is built inside model
# forwards so autograd sees dCDH/dT; see src/models/common.py)
import pandas as pd
cdh = (df['Temperature(2m)'] - dp.CDH_BASE).clip(lower=0)
pd.DataFrame({'cdh': cdh, 'demand': df['demand_clean']}).corr()

In [ ]:
# regenerate all EDA figures into results/figures/
import eda
eda.main()